# Midden-laag spanning Stedin met maximale waterdiepte

In [1]:
from pathlib import Path
import geopandas as gpd
import folium 
from shapely.geometry import LineString, Polygon, box
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import shape

import numpy as np
import geopandas as gpd
from scipy.spatial import Voronoi
from shapely.geometry import Polygon

import rasterio
import geopandas as gpd
from shapely.geometry import Point
from rasterio.transform import Affine
import lonboard as lb

In [2]:
root_dir = Path(r'N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse')
static_path = root_dir.joinpath("static_maxwd")
network_path = static_path.joinpath("network")
stedin_data = root_dir.joinpath("Stedin_data")
hazard_path = static_path.joinpath("hazard")
resultaten_path = static_path.joinpath("resultaten")
assert root_dir.exists()

In [3]:
msls_station_path = stedin_data.joinpath("MiddenLaagspanningsstations")
msls_stations = gpd.read_file(msls_station_path.joinpath("MiddenLaagspanningsstations.shp"), driver='ESRI Shapefile')
msls_stations_centroids = msls_stations['geometry'].representative_point()
bounding_box_msls = msls_stations_centroids.unary_union.convex_hull


In [4]:
boundary = gpd.read_file(network_path.joinpath("stedin_area.geojson"),driver='GeoJSON')

In [5]:
study_area = gpd.read_file(network_path.joinpath("study_area_correct_geometry.gpkg"),driver='GPKG')


In [6]:
study_area

,gml_id,nationalCo,localId,namespace,nationalLe,national_1,country,name,geometry
0,NL.WBHCODE.40.Admingrenswaterschap.1,40,Admingrenswaterschap.1,NL.WBHCODE.40,4thOrder,Waterschap,NL,Noord-Westelijke Delta,"MULTIPOLYGON (((102132.491 433603.397, 102146...."


In [7]:
def voronoi_polygons(gdf, bounding_box_gdf, boundary=None):
    from shapely.geometry import Polygon
    from scipy.spatial import Voronoi
    import geopandas as gpd

    # Combineer geometrieën in bounding_box_gdf tot één polygon
    bounding_geom = bounding_box_gdf.unary_union

    # Genereer Voronoi-diagram
    points = gdf.geometry.apply(lambda geom: (geom.x, geom.y)).tolist()
    vor = Voronoi(points)
    polygons = []

    for region in vor.regions:
        if not -1 in region and len(region) > 0:
            polygon = Polygon([vor.vertices[i] for i in region])
            clipped_polygon = polygon.intersection(bounding_geom)
            polygons.append(clipped_polygon)

    voronoi_gdf = gpd.GeoDataFrame(geometry=polygons, crs=gdf.crs)

    # Clip met boundary als die is meegegeven
    if boundary is not None:
        voronoi_gdf = voronoi_gdf.clip(boundary)

    return voronoi_gdf


# Assuming 'gdf' is your GeoDataFrame with point geometries

gdf_msls = msls_stations_centroids.set_crs("EPSG:28992")
voronoi_gdf_msls = voronoi_polygons(gdf_msls, study_area, boundary=boundary)
voronoi_gdf_msls = voronoi_gdf_msls.set_crs("EPSG:28992")




In [8]:
voronoi_gdf_msls.to_file(resultaten_path.joinpath("voronoi_gdf_msls.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 9887 WHERE lower(table_name) = lower('voronoi_gdf_msls')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 9887 WHERE lower(table_name) = lower('voronoi_gdf_msls')) failed: unable to open database file"


In [9]:
import rasterio
import matplotlib.pyplot as plt

hazard_map = hazard_path.joinpath("RMM_merge_max_waterdepth.tif")
# Open the TIF file using rasterio
with rasterio.open(hazard_map) as src:
    # Read the TIF file as a numpy array
    tif_array = src.read(1)  # Change the band index (1) if necessary


hazard_map_duration = hazard_path.joinpath("RMM_merge_duurkaart_uur.tif")
with rasterio.open(hazard_map_duration) as src:
    # Read the TIF file as a numpy array
    tif_array = src.read(1)  # Change the band index (1) if necessary


In [10]:
# Assuming gdf_msls is already a GeoDataFrame or a DataFrame with geometry data
gdf_msls_2 = gpd.GeoDataFrame()
gdf_msls_2['geometry'] = gdf_msls.geometry

C:\Users\meije_le\AppData\Local\Temp\ipykernel_1616\219393971.py:3: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  gdf_msls_2['geometry'] = gdf_msls.geometry


In [11]:
import rasterio
import geopandas as gpd

# Ensure CRS match between points and study area
points = gdf_msls_2.copy()
study_area = study_area.copy()

# Clip points to study area (ensure valid geometry)
study_area = study_area.buffer(0)
points_clipped = gpd.clip(points, study_area)

# Sample hazard map
with rasterio.open(hazard_map) as src_hazard:
    hazard_crs = src_hazard.crs
    points_clipped_hazard = points_clipped.to_crs(hazard_crs)
    coords_hazard = [(geom.x, geom.y) for geom in points_clipped_hazard.geometry]
    hazard_values = list(src_hazard.sample(coords_hazard))
    points_clipped['hazard_value'] = [val[0] if val else None for val in hazard_values]

# Sample duration map
with rasterio.open(hazard_map_duration) as src_duration:
    duration_crs = src_duration.crs
    points_clipped_duration = points_clipped.to_crs(duration_crs)
    coords_duration = [(geom.x, geom.y) for geom in points_clipped_duration.geometry]
    duration_values = list(src_duration.sample(coords_duration))
    points_clipped['duration_value'] = [val[0] if val else None for val in duration_values]

# Result: points_clipped now contains both hazard and duration values


c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


# Damage calculations on point file

In [12]:
def default_damage_ratio_function(hazard_values, coefficients=(0.0468, 0.0077)):
    """Calculate damage ratio from hazard values using linear function"""
    m, n = coefficients
    return m * hazard_values + n
 
points_clipped['damage_ratio'] = default_damage_ratio_function(points_clipped['hazard_value'])

points_clipped['damage'] = points_clipped['damage_ratio'] * 142500 #construction costs

def default_repair_time_function(damage_ratios,coefficients=(702.72, 3.14, 1.9891)):
    """Calculate repair time from damage ratios using polynomial function"""
    a, b, c = coefficients
    return a * (damage_ratios ** 2) + b * damage_ratios + c


points_clipped['repair_time'] = default_repair_time_function(points_clipped['damage_ratio'])
points_clipped["duration_value"]= points_clipped["duration_value"].fillna(0)
points_clipped["duration+repair_time"] = points_clipped["duration_value"].fillna(0) + points_clipped["repair_time"]


c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value

In [13]:
points_clipped.to_file(resultaten_path.joinpath("msls_stations_with_hazard.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 10034 WHERE lower(table_name) = lower('msls_stations_with_hazard')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 10034 WHERE lower(table_name) = lower('msls_stations_with_hazard')) failed: disk I/O error"


# Voronoi calculations

In [14]:
import geopandas as gpd

def assign_hazard_to_voronoi(points_with_hazard, voronoi_gdf_msls):
    # Ensure both GeoDataFrames have the same CRS
    if points_with_hazard.crs != voronoi_gdf_msls.crs:
        points_with_hazard = points_with_hazard.to_crs(voronoi_gdf_msls.crs)
    
    # Spatial join: find Voronoi polygons that contain points
    joined_gdf = gpd.sjoin(voronoi_gdf_msls, points_with_hazard, how="inner", predicate="contains")
    
    # Group by Voronoi polygon and aggregate both hazard and duration values
    aggregated = joined_gdf.groupby(joined_gdf.index).agg({
        'hazard_value': 'mean',
        'duration_value': 'mean'
    }).reset_index()
    
    # Merge aggregated values back to the original Voronoi GeoDataFrame
    voronoi_gdf_msls = voronoi_gdf_msls.merge(aggregated, left_index=True, right_on='index', how='left')
    
    return voronoi_gdf_msls


In [15]:
result = assign_hazard_to_voronoi(points_clipped, voronoi_gdf_msls)
result.to_file(resultaten_path.joinpath("msls_voronoi_hazard.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('msls_voronoi_hazard')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('msls_voronoi_hazard')) failed: disk I/O error"


# Assign population to voronoi

In [16]:
# Load and clip population data
population_data = gpd.read_file(r"N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\network\cbs_population_data_fixed_geometry.gpkg")
population_data = population_data.clip(study_area)

In [17]:

def assign_cell_data_to_voronoi(voronoi_gdf, cell_data_gdf, data_column='cell_value'):
    """
    Assign cell-based data to Voronoi polygons using area-weighted intersection

    Parameters:
    -----------
    voronoi_gdf : GeoDataFrame
        Voronoi polygons with asset_id column
    cell_data_gdf : GeoDataFrame
        Data cells containing spatially distributed values (e.g., population, economic value)
    data_column : str
        Column name containing the cell data values

    Returns:
    --------
    GeoDataFrame : Voronoi polygons with assigned_cell_data column
    """
    import geopandas as gpd

    # Copy inputs
    voronoi_polygons = voronoi_gdf.copy()
    data_cells = cell_data_gdf.copy()

    # Ensure CRS match
    data_cells = data_cells.to_crs(voronoi_polygons.crs)

    # Calculate area of each data cell (A_i)
    data_cells['A_i'] = data_cells.geometry.area

    # Preserve Voronoi index before overlay
    voronoi_polygons['V_index'] = voronoi_polygons.index

    # Spatial intersection to get A_{i ∩ V_j}
    intersections = gpd.overlay(data_cells, voronoi_polygons, how="intersection")

    # Calculate area of intersection
    intersections['A_iV_j'] = intersections.geometry.area

    # Area-weighted data contribution: D_i * (A_{i ∩ V_j} / A_i)
    weighted_string = 'weighted_' + data_column
    intersections[weighted_string] = (
        intersections[data_column] * intersections['A_iV_j'] / intersections['A_i']
    )

    # Aggregate data by Voronoi polygon
    data_by_voronoi = (
        intersections.groupby('V_index')[weighted_string].sum().reset_index()
    )

    # Merge back to Voronoi polygons
    voronoi_polygons = voronoi_polygons.merge(data_by_voronoi, on="V_index", how="left")
    voronoi_polygons[weighted_string] = voronoi_polygons[weighted_string].fillna(0)

    return voronoi_polygons


In [18]:
# Discard all negative values in 'aantal inwoners' (no data values)
population_data = population_data[population_data['aantal_inwoners'] >= 0]
voronoi_impact = assign_cell_data_to_voronoi(result, population_data, data_column='aantal_inwoners')

## Import grid results for accessibility analysis

In [19]:
grid_results_path = Path(r"N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\output_graph\grid_analysis_V_K_analyse_Rijn-Maasmonding_0_1me-0_25fr_20251103_155048_wduration.parquet")
grid_results = gpd.read_parquet(grid_results_path)

In [20]:
grid_results['%_difference_reachable'] = (
    (grid_results['reachable_cells_no_flood'] - grid_results['reachable_cells_hazard']) 
    / grid_results['reachable_cells_no_flood'].replace(0, float('nan')) * 100
)

# Ensure the active geometry column is the original one
grid_results = grid_results.set_geometry("geometry")  # Replace with your original geometry column name

# Drop the centroid column (or convert it to WKT if you need to keep the info)
grid_results = grid_results.drop(columns=["centroid"])  # Replace with actual centroid column name

# Save to GeoPackage
grid_results.to_file(resultaten_path.joinpath("grid_results_with_%_diff.gpkg"), driver="GPKG")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('grid_results_with_%_diff')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('grid_results_with_%_diff')) failed: unable to open database file"


In [21]:
grid_results.to_file(resultaten_path.joinpath("grid_results_with_%_diff.gpkg"), driver='GPKG')

DriverError: A file system object called 'N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\resultaten\grid_results_with_%_diff.gpkg' already exists.

#### Relate this to the electricity stations

In [ ]:
# After your spatial join
points_with_grid = gpd.sjoin(points_clipped, grid_results, how="left", predicate="within")

# ✅ Use the geometry from points_clipped
points_with_grid = points_with_grid.set_geometry(points_clipped.geometry)


✅ Exported points_with_grid with point geometry to N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\resultaten


In [ ]:
points_with_grid["diff_no_acces_flood_substations_duration"] = (
    points_with_grid["duration_value"] - points_with_grid["max_duration"]
)

# Step 1: Maximum of duration_value and max_duration
points_with_grid["max_duration_combined"] = np.maximum(
    points_with_grid["duration_value"],
    points_with_grid["max_duration"]
)

# Step 2: Add repair time to the maximum
points_with_grid["max_duration+repair_time"] = (
    points_with_grid["max_duration_combined"] + points_with_grid["repair time"]
)

points_with_grid["determinant_factor"] = np.where(
    points_with_grid["duration_value"] > points_with_grid["max_duration"],
    "Het onderstation is langer overstroomd dan ontoegankelijk",
    np.where(
        points_with_grid["max_duration"] > points_with_grid["duration_value"],
        "Het onderstation is langer ontoegankelijk dan overstroomd",
        "Beide duren even lang"
    )
)

In [ ]:
# ✅ Export directly to GeoPackage
output_path = resultaten_path
points_with_grid.to_file(output_path.joinpath("stations_reachability.gpkg"), driver='GPKG')

print(f"✅ Exported points_with_grid with point geometry to {output_path}")

# Total person loss hours

In [22]:
voronoi_impacts = gpd.read_file(r"N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\resultaten\msls_voronoi_impact.gpkg")

In [23]:
voronoi_impacts["person_loss_hours"] = voronoi_impacts["assigned_impact_metric"] * voronoi_impacts["duration_value"]
voronoi_impacts = voronoi_impacts[voronoi_impacts["assigned_impact_metric"] > 0]

In [24]:
voronoi_impacts.to_file(resultaten_path.joinpath("voronoi_impact_person_losshours.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('voronoi_impact_person_losshours')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('voronoi_impact_person_losshours')) failed: unable to open database file"


## Impact per municipality

In [ ]:
municipality = gpd.read_file(network_path.joinpath("gemeentegrenzen.shp"),driver='Esri Shapefile')
municipality_study_area = municipality.clip(study_area)

In [25]:
wijkkaarten = gpd.read_file(network_path.joinpath("wijken_2022_v2.shp"),driver='Esri Shapefile')
wijkkaarten_study_area = wijkkaarten.clip(study_area)

In [26]:
wijkkaarten_study_area = wijkkaarten_study_area[wijkkaarten_study_area["WK_NAAM"] != "Groot water"]

# New code redistributes the person loss hours per wijk

In [47]:

import geopandas as gpd
import pandas as pd
import numpy as np

def redistribute_ph_and_pct_affected(
    wijk_gdf: gpd.GeoDataFrame,
    voronoi_gdf: gpd.GeoDataFrame,
    ph_col: str = 'person_loss_hours',           # existing PH on Voronoi
    population_col: str = 'assigned_impact_metric',  # base population per Voronoi
    wijk_population_col: str = 'AANT_INW',       # total population per wijk
    hazard_col: str = 'hazard_value',
    hazard_threshold: float = 0.45,              # only include Voronoi with hazard > threshold
    wijk_name_col: str = 'WK_NAAM',
    min_overlap_m2: float = 20,                  # drop micro-slivers (set 0 to disable)
    target_epsg: int = 28992                     # NL meter CRS for area math
) -> gpd.GeoDataFrame:
    """
    Filter Voronoi cells by hazard_value > threshold, then area-redistribute to wijken:
      - person_loss_hours  -> summed per wijk
      - impacted_inwoners  -> summed per wijk (from base Voronoi population)
      - perc_impacted_inwoners = impacted_inwoners / AANT_INW * 100

    No duration/hazard recalculation—just filtering + areal weighting.

    Returns a GeoDataFrame with one row per wijk that includes:
      [WK_NAAM, geometry, person_loss_hours, impacted_inwoners, AANT_INW, perc_impacted_inwoners]
    """

    # --- Copies to avoid side effects
    wijk = wijk_gdf.copy()
    voro = voronoi_gdf.copy()

    # --- Validate columns
    for c in [ph_col, population_col, hazard_col]:
        if c not in voro.columns:
            raise KeyError(f"'{c}' not found on voronoi_gdf.")
    if wijk_population_col not in wijk.columns:
        raise KeyError(f"'{wijk_population_col}' not found on wijk_gdf.")
    if wijk_name_col not in wijk.columns:
        # Fall back to a name from index if needed
        wijk[wijk_name_col] = wijk.index.astype(str)

    # --- Project to a meter-based CRS for area computations
    def _to_meters(gdf):
        if gdf.crs is None:
            gdf = gdf.set_crs(target_epsg)
        else:
            gdf = gdf.to_crs(target_epsg)
        return gdf

    wijk = _to_meters(wijk)
    voro = _to_meters(voro)

    # --- Repair geometry issues
    wijk['geometry'] = wijk.buffer(0)
    voro['geometry'] = voro.buffer(0)

    # --- Geometry type check
    if not all(voro.geom_type.isin(['Polygon', 'MultiPolygon'])):
        raise TypeError("voronoi_gdf must contain Polygon/MultiPolygon geometries.")

    # --- Filter Voronoi cells by hazard threshold
    voro[hazard_col] = pd.to_numeric(voro[hazard_col], errors='coerce')
    eligible = voro[voro[hazard_col] > hazard_threshold].copy()

    # If nothing passes the threshold → return zeros per wijk
    if eligible.empty:
        wijk['person_loss_hours'] = 0.0 if ph_col == 'person_loss_hours' else 0.0
        wijk['impacted_inwoners'] = 0.0
        wijk['perc_impacted_inwoners'] = 0.0
        return wijk[[wijk_name_col, 'geometry', 'person_loss_hours', 'impacted_inwoners',
                     wijk_population_col, 'perc_impacted_inwoners']]

    # --- Precompute areas & stable wijk index
    eligible['voronoi_area'] = eligible.geometry.area
    eligible.loc[eligible['voronoi_area'] <= 0, 'voronoi_area'] = np.nan
    wijk['wijk_index'] = wijk.index

    # --- Intersections (true overlaps)
    intersections = gpd.overlay(
        eligible,
        wijk[['wijk_index', 'geometry']],
        how='intersection'
    )

    if intersections.empty:
        wijk['person_loss_hours'] = 0.0 if ph_col == 'person_loss_hours' else 0.0
        wijk['impacted_inwoners'] = 0.0
        wijk['perc_impacted_inwoners'] = 0.0
        return wijk[[wijk_name_col, 'geometry', 'person_loss_hours', 'impacted_inwoners',
                     wijk_population_col, 'perc_impacted_inwoners']]

    # --- Intersection area & optional sliver filter
    intersections['overlap_area'] = intersections.geometry.area
    if min_overlap_m2 and min_overlap_m2 > 0:
        before = len(intersections)
        intersections = intersections.loc[intersections['overlap_area'] >= float(min_overlap_m2)].copy()
        after = len(intersections)
        if before != after:
            print(f"ℹ Dropped {before - after} micro-intersections (< {min_overlap_m2} m²).")
        if intersections.empty:
            wijk['person_loss_hours'] = 0.0 if ph_col == 'person_loss_hours' else 0.0
            wijk['impacted_inwoners'] = 0.0
            wijk['perc_impacted_inwoners'] = 0.0
            return wijk[[wijk_name_col, 'geometry', 'person_loss_hours', 'impacted_inwoners',
                         wijk_population_col, 'perc_impacted_inwoners']]

    # --- Areal weights
    intersections['voronoi_area'] = intersections['voronoi_area'].replace(0, np.nan)
    intersections['w'] = intersections['overlap_area'] / intersections['voronoi_area']

    # --- Redistribute PH (coerce to numeric; NaN→0)
    intersections[ph_col] = pd.to_numeric(intersections[ph_col], errors='coerce').fillna(0.0)
    intersections['redist_ph'] = intersections[ph_col] * intersections['w']

    # --- Redistribute impacted population (from base Voronoi population)
    intersections[population_col] = pd.to_numeric(intersections[population_col], errors='coerce').fillna(0.0)
    intersections['redist_impacted'] = intersections[population_col] * intersections['w']

    # --- Aggregate per wijk
    agg = (intersections
           .groupby('wijk_index', as_index=False)[['redist_ph', 'redist_impacted']]
           .sum()
           .rename(columns={'redist_ph': 'person_loss_hours',
                            'redist_impacted': 'impacted_inwoners'}))

    # --- Merge back to wijken
    wijk_out = wijk.merge(agg, on='wijk_index', how='left')
    wijk_out['person_loss_hours'] = wijk_out['person_loss_hours'].fillna(0.0)
    wijk_out['impacted_inwoners'] = wijk_out['impacted_inwoners'].fillna(0.0)

    # --- Percentage affected
    denom = wijk_out[wijk_population_col].replace(0, np.nan)
    wijk_out['perc_impacted_inwoners'] = ((wijk_out['impacted_inwoners'] / denom) * 100).round(2).fillna(0.0)

    # --- (Optional) Conservation checks over the eligible subset
    total_eligible_ph = pd.to_numeric(eligible[ph_col], errors='coerce').fillna(0.0).sum()
    total_wijk_ph = wijk_out['person_loss_hours'].sum()
    if abs(total_wijk_ph - total_eligible_ph) > 1e-6:
        print(f"⚠ PH conservation (eligible subset): wijk - voronoi = {total_wijk_ph - total_eligible_ph:.6f}")

    total_eligible_pop = pd.to_numeric(eligible[population_col], errors='coerce').fillna(0.0).sum()
    total_wijk_pop = wijk_out['impacted_inwoners'].sum()
    if abs(total_wijk_pop - total_eligible_pop) > 1e-6:
        print(f"⚠ Pop conservation (eligible subset): wijk - voronoi = {total_wijk_pop - total_eligible_pop:.6f}")

    # --- Final columns to return
    return wijk_out[
        [wijk_name_col, 'geometry',
         'person_loss_hours', 'impacted_inwoners', wijk_population_col, 'perc_impacted_inwoners']
    ]


In [48]:

wijk_stats = redistribute_ph_and_pct_affected(
    wijk_gdf=wijkkaarten_study_area,
    voronoi_gdf=voronoi_impacts,
    ph_col='person_loss_hours',              # already on Voronoi
    population_col='assigned_impact_metric', # base population per Voronoi cell
    wijk_population_col='AANT_INW',
    hazard_col='hazard_value',
    hazard_threshold=0.45,
    wijk_name_col='WK_NAAM',
    min_overlap_m2=20
)

print(wijk_stats[['WK_NAAM','person_loss_hours','impacted_inwoners','AANT_INW','perc_impacted_inwoners']].head())


ℹ Dropped 3 micro-intersections (< 20 m²).
⚠ PH conservation (eligible subset): wijk - voronoi = -309.899862
⚠ Pop conservation (eligible subset): wijk - voronoi = -2.590681
                        WK_NAAM  person_loss_hours  impacted_inwoners  \
0                  Lage Zwaluwe                0.0                0.0   
1  Wijk 98 Verspreide bebouwing                0.0                0.0   
2             Boven Hardinxveld                0.0                0.0   
3                     Werkendam                0.0                0.0   
4             Wijk 10 Dubbeldam                0.0                0.0   

   AANT_INW  perc_impacted_inwoners  
0      4220                     0.0  
1       525                     0.0  
2      4670                     0.0  
3     11280                     0.0  
4     12815                     0.0  


In [49]:
wijk_stats.to_file(resultaten_path.joinpath("wijk_stats_hazard_duration.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('wijk_stats_hazard_duration')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('wijk_stats_hazard_duration')) failed: unable to open database file"


### OLD CODE that  recalculates per wijk maximum duratoin found * inwoners getroffen

Per wijk:
Is er overstroming ja of nee en wat is maximale duur?
Is er stroomuitval en wat is de duur? hoeveel mensen geraakt
Is er verminderde bereikbaarheid?

In [ ]:
import geopandas as gpd

def redistribute_population_to_wijk_old(
    wijk_gdf,
    voronoi_gdf,
    hazard_col='hazard_value',
    duration_col='flood_duration_substation',  # duration per substation
    wijk_name_col='WK_NAAM',
    population_col='assigned_impact_metric',
    wijk_population_col='AANT_INW'
):
    """
    Redistribute Voronoi population to wijken based on area-weighted overlap.
    Calculates:
    - max hazard per wijk
    - max_flood_duration_per_wijk (maximum flooding duration of substations intersecting the wijk)
    - impacted inwoners if hazard > 0.2 and > 0.45
    - % impacted population for both thresholds relative to AANT_INW
    - person-hours per wijk for both thresholds:
      person_hours_gt_0_2 = inwoners_hazard_gt_0_2 * max_flood_duration_per_wijk
      person_hours_gt_0_45 = inwoners_hazard_gt_0_45 * max_flood_duration_per_wijk
      BUT if max_flood_duration_per_wijk <= 0 → use inwoners count instead.
    """
    # Ensure CRS match
    voronoi_gdf = voronoi_gdf.to_crs(wijk_gdf.crs)

    # Compute Voronoi area
    voronoi_gdf['voronoi_area'] = voronoi_gdf.geometry.area

    # Preserve wijk index
    wijk_gdf['wijk_index'] = wijk_gdf.index

    # Overlay to compute intersections
    intersections = gpd.overlay(voronoi_gdf, wijk_gdf, how='intersection')

    if intersections.empty:
        print("⚠ Geen intersecties gevonden.")
        for col in ['max_hazard', 'max_flood_duration_per_wijk', 'inwoners_hazard_gt_0_2', 'inwoners_hazard_gt_0_45',
                    'perc_impacted_inwoners_0_2', 'perc_impacted_inwoners_0_45',
                    'person_hours_gt_0_2', 'person_hours_gt_0_45']:
            wijk_gdf[col] = 0
        return wijk_gdf[[wijk_name_col, 'geometry', 'max_hazard', 'max_flood_duration_per_wijk',
                         'inwoners_hazard_gt_0_2', 'inwoners_hazard_gt_0_45',
                         'perc_impacted_inwoners_0_2', 'perc_impacted_inwoners_0_45',
                         'person_hours_gt_0_2', 'person_hours_gt_0_45', wijk_population_col]]

    # Compute intersection area
    intersections['overlap_area'] = intersections.geometry.area

    # Compute proportional population contribution
    intersections['redistributed_inwoners'] = (
        intersections[population_col] * intersections['overlap_area'] / intersections['voronoi_area']
    )

    # Apply hazard thresholds for inwoners
    intersections['inwoners_hazard_gt_0_2'] = intersections.apply(
        lambda row: row['redistributed_inwoners'] if row[hazard_col] > 0.2 else 0, axis=1
    )
    intersections['inwoners_hazard_gt_0_45'] = intersections.apply(
        lambda row: row['redistributed_inwoners'] if row[hazard_col] > 0.45 else 0, axis=1
    )

    # Aggregate by wijk
    agg = intersections.groupby('wijk_index').agg({
        hazard_col: 'max',
        duration_col: 'max',  # max flooding duration across substations
        'inwoners_hazard_gt_0_2': 'sum',
        'inwoners_hazard_gt_0_45': 'sum'
    }).reset_index()

    agg.rename(columns={
        hazard_col: 'max_hazard',
        duration_col: 'max_flood_duration_per_wijk'  # ✅ clearer name
    }, inplace=True)

    # Merge back to wijken
    wijk_gdf = wijk_gdf.merge(agg, on='wijk_index', how='left')

    # Fill NaN for non-overlapping wijken
    for col in ['max_hazard', 'max_flood_duration_per_wijk', 'inwoners_hazard_gt_0_2', 'inwoners_hazard_gt_0_45']:
        wijk_gdf[col] = wijk_gdf[col].fillna(0)

    # ✅ Compute person-hours per wijk using rule:
    # If max_flood_duration_per_wijk <= 0 → use inwoners count instead of multiplying
    wijk_gdf['person_hours_gt_0_2'] = wijk_gdf.apply(
        lambda row: row['inwoners_hazard_gt_0_2'] if row['max_flood_duration_per_wijk'] <= 0
        else row['inwoners_hazard_gt_0_2'] * row['max_flood_duration_per_wijk'], axis=1
    )
    wijk_gdf['person_hours_gt_0_45'] = wijk_gdf.apply(
        lambda row: row['inwoners_hazard_gt_0_45'] if row['max_flood_duration_per_wijk'] <= 0
        else row['inwoners_hazard_gt_0_45'] * row['max_flood_duration_per_wijk'], axis=1
    )

    # Calculate percentages for both thresholds
    wijk_gdf['perc_impacted_inwoners_0_2'] = (
        (wijk_gdf['inwoners_hazard_gt_0_2'] / wijk_gdf[wijk_population_col].replace(0, float('nan'))) * 100
    ).round(2).fillna(0)
    wijk_gdf['perc_impacted_inwoners_0_45'] = (
        (wijk_gdf['inwoners_hazard_gt_0_45'] / wijk_gdf[wijk_population_col].replace(0, float('nan'))) * 100
    ).round(2).fillna(0)

    # Debug prints
    print("✅ Intersecties:", len(intersections))
    print(f"Som inwoners hazard > 0.2: {wijk_gdf['inwoners_hazard_gt_0_2'].sum()}")
    print(f"Som inwoners hazard > 0.45: {wijk_gdf['inwoners_hazard_gt_0_45'].sum()}")
    print(f"Som person-hours hazard > 0.2: {wijk_gdf['person_hours_gt_0_2'].sum()}")
    print(f"Som person-hours hazard > 0.45: {wijk_gdf['person_hours_gt_0_45'].sum()}")

    # ✅ Return per wijk values
    return wijk_gdf[[wijk_name_col, 'geometry', 'max_hazard', 'max_flood_duration_per_wijk',
                     'inwoners_hazard_gt_0_2', 'inwoners_hazard_gt_0_45',
                     'perc_impacted_inwoners_0_2', 'perc_impacted_inwoners_0_45',
                     'person_hours_gt_0_2', 'person_hours_gt_0_45',
                     wijk_population_col]]

In [ ]:
# wijk_stats = redistribute_population_to_wijk(wijkkaarten_study_area, voronoi_impacts, hazard_col='hazard_value', duration_col='duration_value')


### Wegen

In [ ]:
wegen = gpd.read_file(r"N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\output_graph\base_graph_hazard_edges.gpkg", drivr='GPKG')

In [ ]:
import geopandas as gpd

def calculate_flooded_road_percentage(wijk_gdf, roads_gdf, wijk_name_col='WK_NAAM'):
    """
    Calculate % of road length per wijk that is flooded based on EV2 thresholds.
    """
    # Ensure CRS match and in meters
    roads_gdf = roads_gdf.to_crs(wijk_gdf.crs)

    # Filter flooded roads
    flooded_roads = roads_gdf[(roads_gdf['EV2_me'] > 0.2) & (roads_gdf['EV2_fr'] > 0.25)]

    # Compute lengths
    roads_gdf['road_length'] = roads_gdf.geometry.length
    flooded_roads['road_length'] = flooded_roads.geometry.length

    # Spatial join: assign wijk to each road
    roads_joined = gpd.sjoin(roads_gdf, wijk_gdf, how='inner', predicate='intersects')
    flooded_joined = gpd.sjoin(flooded_roads, wijk_gdf, how='inner', predicate='intersects')

    # Aggregate lengths per wijk
    total_length = roads_joined.groupby('index_right')['road_length'].sum()
    flooded_length = flooded_joined.groupby('index_right')['road_length'].sum()

    # Merge into wijk_gdf
    wijk_gdf['total_road_length'] = wijk_gdf.index.map(total_length).fillna(0)
    wijk_gdf['flooded_road_length'] = wijk_gdf.index.map(flooded_length).fillna(0)

    # Calculate percentage
    wijk_gdf['perc_flooded_roads'] = (
        (wijk_gdf['flooded_road_length'] / wijk_gdf['total_road_length']) * 100
    ).round(2)

    # Keep relevant columns
    return wijk_gdf[[wijk_name_col, 'geometry', 'total_road_length', 'flooded_road_length', 'perc_flooded_roads']]

In [ ]:
wijk_wegen = calculate_flooded_road_percentage(wijkkaarten_study_area, wegen, wijk_name_col='WK_NAAM')

c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
wijk_wegen.to_file(resultaten_path.joinpath("wijk_wegen_flooded_percentage.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_wijk_wegen_flooded_percentage" AFTER DELETE ON "wijk_wegen_flooded_percentage" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'wijk_wegen_flooded_percentage\'); END;) failed: unable to open database file'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_wijk_wegen_flooded_percentage" AFTER DELETE ON "wijk_wegen_flooded_percentage" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'wijk_wegen_flooded_percentage\'); END;) failed: unable to open database file'


# 